# 🗣️ 01 · 传统语音识别：GMM-HMM 与 DTW

00 讲了**特征**（每帧一行向量），本节讲两个「如何利用特征」的经典框架：

- **DTW**：不建模型，模板与测试序列直接**时间对齐**匹配；
- **GMM-HMM**：为「词/音素 → 特征帧」建立**概率生成模型**，用 Viterbi 解码。

两者都服务于同一个核心痛点：**语音长度可变、语速不同，帧对不上**。

## 📋 本节要解决的问题

| # | 问题 | 框架 |
|---|---|---|
| 1 | 两段语音帧数不同怎么比？ | DTW 动态规划对齐 |
| 2 | 一条路径的总概率怎么算？ | HMM 前向算法 |
| 3 | 最可能的隐藏状态序列？ | Viterbi（max 版前向） |
| 4 | 参数怎么从数据学出来？ | Baum-Welch = EM |
| 5 | 连续 MFCC 特征怎么建模发射概率？ | GMM 混合高斯 |
| 6 | 整句/整词怎么打分与解码？ | 前向打分 + Viterbi 解码 |

## 1️⃣ 隐马尔可夫模型：语音的生成故事

想象说话 = 背后一条**状态链**（音素/拼音）在生成特征帧；我们只看得到帧（观察），看不到状态（隐藏）。

**HMM λ = (A, B, π)**：

- $A = [a_{ij}]$：状态转移概率 $P(q_{t+1}=j \mid q_t=i)$；
- $B = [b_j(o)]$：发射概率（状态 $j$ 生成观察 $o$）；
- $\pi = [\pi_i]$：初始状态分布。

两条关键假设：**齐次性**（转移与 $t$ 无关）、**观测独立**（$o_t$ 只依赖 $q_t$）。    
三个经典问题：**求值**（P(O|λ)，前向）、**解码**（最优状态链，Viterbi）、**学习**（λ，Baum-Welch）。

## 2️⃣ 前向算法：P(O|λ) 高效计算

直接枚举所有状态链是 $O(N^T)$，指数爆炸。前向算法用 DP：

定义 $\alpha_t(i) = P(o_1,\dots,o_t,\ q_t=i \mid \lambda)$，递推：

$$ \alpha_{t+1}(j) = \left[\sum_{i=1}^{N} \alpha_t(i)\, a_{ij}\right] \cdot b_j(o_{t+1}) $$

终止：$P(O\mid\lambda)=\sum_i \alpha_T(i)$。复杂度降到 $O(TN^2)$。

> 数值：概率连乘下溢 → 全部在 **log 域**计算，用 `logsumexp` 做 log 下的求和。

In [1]:
import numpy as np, matplotlib.pyplot as plt
import itertools
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(0)

def logsumexp(a, axis=0):
    amax = a.max(axis=axis, keepdims=True)
    return np.log(np.exp(a - amax).sum(axis=axis)) + amax.squeeze(axis)

def forward_log(A, B, pi):
    """A:(N,N) log转移, B:(T,N) log发射, pi:(N,) log初始 → (log P(O|λ), alpha表)"""
    T, N = B.shape
    alpha = np.full((T, N), -np.inf)
    alpha[0] = pi + B[0]
    for t in range(1, T):
        for j in range(N):
            alpha[t, j] = logsumexp(alpha[t-1] + A[:, j]) + B[t, j]
    return logsumexp(alpha[-1]), alpha

N, V, T = 2, 3, 6
A = np.log([[0.7, 0.3], [0.4, 0.6]])
pi = np.log([0.6, 0.4])
emis = np.array([[0.6, 0.3, 0.1], [0.1, 0.3, 0.6]])   # (状态, 符号)
obs = np.array([0, 1, 2, 1, 0, 2])
B = np.log(emis.T[obs])                                   # (T, N) log 发射
loglik, alpha = forward_log(A, B, pi)
print('log P(O|λ) = %.4f  →  P = %.6f' % (loglik, np.exp(loglik)))
print('alpha 表 (T×N):\n', np.round(alpha, 3))

plt.show()


log P(O|λ) = -6.8406  →  P = 0.001069
alpha 表 (T×N):
 [[-1.022 -3.219]
 [-2.521 -3.229]
 [-4.932 -3.55 ]
 [-5.306 -5.146]
 [-5.66  -7.605]
 [-8.241 -7.124]]


In [2]:
def brute_force(A, B, pi):
    T, N = B.shape
    total = 0.0
    for path in itertools.product(range(N), repeat=T):
        p = pi[path[0]] + B[0, path[0]]
        for t in range(1, T):
            p += A[path[t-1], path[t]] + B[t, path[t]]
        total += np.exp(p)
    return np.log(total)

val = brute_force(A, B, pi)
print(f'穷举 {N**T} 条路径 logP = {val:.4f} | 前向 = {loglik:.4f}')
assert abs(val - loglik) < 1e-6, '前向算法数值错误'
print('✓ 前向算法与路径穷举一致')

穷举 64 条路径 logP = -6.8406 | 前向 = -6.8406
✓ 前向算法与路径穷举一致


## 3️⃣ Viterbi：找最可能的隐藏状态链

前向是**求和**（所有路径），Viterbi 是**取 max**（一条最优路径）：

$$ \delta_{t}(j) = \max_{q_1\dots q_{t-1}} P(q_1\dots q_{t-1}, q_t=j, o_1\dots o_t\mid\lambda) $$

$$ \delta_t(j) = \max_i\left[\delta_{t-1}(i)\, a_{ij}\right] b_j(o_t), \qquad \psi_t(j) = \arg\max_i [\delta_{t-1}(i) a_{ij}] $$

$\psi$ 记录**回溯指针**，结尾取 $\arg\max_j \delta_T(j)$ 后沿指针回走得到整条链。

> 面试一句话：**前向算概率（所有路），Viterbi 算路径（最优路）；都吃 log 域防下溢**。

In [3]:
def viterbi_log(A, B, pi):
    T, N = B.shape
    delta = np.full((T, N), -np.inf)
    psi = np.zeros((T, N), dtype=int)
    delta[0] = pi + B[0]
    for t in range(1, T):
        for j in range(N):
            tmp = delta[t-1] + A[:, j]
            psi[t, j] = np.argmax(tmp)
            delta[t, j] = tmp[psi[t, j]] + B[t, j]
    path = np.zeros(T, dtype=int)
    path[-1] = np.argmax(delta[-1])
    for t in range(T-1, 0, -1):
        path[t-1] = psi[t, path[t]]
    return path, delta[-1].max()

path, lp = viterbi_log(A, B, pi)
best, best_score = None, -np.inf
for cand in itertools.product(range(N), repeat=T):
    p = pi[cand[0]] + B[0, cand[0]]
    for t in range(1, T):
        p += A[cand[t-1], cand[t]] + B[t, cand[t]]
    if p > best_score:
        best, best_score = cand, p
print('Viterbi 路径 =', path, ' logP* = %.4f' % lp)
print('穷举最优     =', np.array(best), ' logP* = %.4f' % best_score)
assert np.array_equal(path, np.array(best)), 'Viterbi 应等于穷举最优'
print('✓ 与穷举最优路径一致')

Viterbi 路径 = [0 0 1 0 0 1]  logP* = -8.9997
穷举最优     = [0 0 1 0 0 1]  logP* = -8.9997
✓ 与穷举最优路径一致


## 4️⃣ Baum-Welch：用 EM 学参数

没有标注状态链，只有观察序列 → **EM（期望最大化）**框架：

- **E 步**：用当前 λ 算后验。前向 $\alpha_t(i)$、后向 $\beta_t(i)$，得
  - $\gamma_t(i) = \dfrac{\alpha_t(i)\beta_t(i)}{P(O\mid\lambda)}$：时刻 $t$ 处于状态 $i$ 的概率；
  - $\xi_t(i,j) = \dfrac{\alpha_t(i) a_{ij} b_j(o_{t+1}) \beta_{t+1}(j)}{P(O\mid\lambda)}$：时刻 $t\to t+1$ 走边 $i\to j$ 的概率。
- **M 步**：按**计数归一化**重估：$\hat a_{ij} = \frac{\sum_t \xi_t(i,j)}{\sum_t \gamma_t(i)}$，$\hat\pi_i = \gamma_1(i)$，GMM 参数同理重估。

迭代至对数似然收敛。**局部最优**是 EM 的固有风险 → 多随机初值。

## 5️⃣ GMM：连续特征的发射概率

MFCC 是连续向量，离散符号表不适用 → 用**高斯混合模型**：

$$ b_j(o) = \sum_{m=1}^{M} w_{jm}\, \mathcal{N}(o; \mu_{jm}, \Sigma_{jm}), \qquad |w|=1 $$

- $M$ 个高斯分量，权重 $w$ 归一；语音常用**对角协方差**（省参数、稳）。
- log 域计算：$\log b_j(o) = \mathrm{logsumexp}_m\big(\log w_{jm} + \log\mathcal{N}\big)$。
- 单个元音 ≈ 某个 GMM；一段话 = 状态序列依次发出各帧。

In [4]:
def gmm_emission(o, weights, means, covs):
    """o:(D,) 观测, weights:(M,), means:(M,D), covs:(M,D)对角方差 → log b(o)"""
    M = len(weights)
    logg = np.empty(M)
    for m in range(M):
        var = covs[m]
        diff = o - means[m]
        logg[m] = (-0.5*np.sum(np.log(2*np.pi*var))
                   - 0.5*np.sum(diff**2/var))
    return logsumexp(np.log(weights) + logg)

np.random.seed(7)
weights = np.array([0.4, 0.6])
means = np.array([[0.0, 0.0], [3.0, 3.0]])
covs = np.array([[0.3, 0.5], [0.4, 0.2]])
o = np.array([1.2, 0.8])
logv = gmm_emission(o, weights, means, covs)
direct = sum(w*np.exp(-0.5*np.sum((o-m)**2/v))
             / np.sqrt((2*np.pi)**2*np.prod(v))
             for w, m, v in zip(weights, means, covs))
print('log 域 = %.6f | 直接 = %.6f | 误差 = %.2e' % (logv, np.log(direct), abs(logv-np.log(direct))))
assert abs(logv - np.log(direct)) < 1e-9
print('✓ GMM 发射 log 域计算正确')

log 域 = -4.845604 | 直接 = -4.845604 | 误差 = 0.00e+00
✓ GMM 发射 log 域计算正确


In [5]:
np.random.seed(42)
N, M, D = 2, 2, 2
means = np.array([[[0, 0], [1, 1]], [[4, 4], [3, 3]]], dtype=float)  # 状态0≈(0,0) 状态1≈(4,4)
covs = np.full((N, M, D), 0.5)
weights = np.full((N, M), 0.5)
obs = np.vstack([np.random.randn(6, 2), 4 + np.random.randn(6, 2)])  # 先状态0后状态1
A = np.log([[0.8, 0.2], [0.2, 0.8]])
pi = np.log([0.5, 0.5])
B = np.array([[gmm_emission(o, weights[s], means[s], covs[s]) for s in range(N)] for o in obs])
loglik, _ = forward_log(A, B, pi)
path, _ = viterbi_log(A, B, pi)
print('整句 log P(O|λ) = %.3f' % loglik)
print('Viterbi 状态：前 6 帧 →', path[:6], ' 后 6 帧 →', path[6:])
print('✓ 期望：前 6 帧状态 0（均值≈0），后 6 帧状态 1（均值≈4）')

整句 log P(O|λ) = -34.773
Viterbi 状态：前 6 帧 → [0 0 0 0 0 0]  后 6 帧 → [1 1 1 1 1 1]
✓ 期望：前 6 帧状态 0（均值≈0），后 6 帧状态 1（均值≈4）


## 6️⃣ DTW：不用统计模型的时间对齐

场景：模板语音 30 帧、测试语音 12 帧（说得快）——逐帧欧氏距离必然偏大，因为**时间没对上**。

**动态时间规整**（DTW）用动态规划找一条**对齐路径**：

- 每步允许三种移动：右（跳 b）、上（跳 a）、对角（对齐一对）；
- 累积代价最小 = 最优对齐；路径单调、边界固定（首尾对齐）。

复杂度 $O(T_1 T_2)$。DTW 与 HMM 的关系：**DTW ≈ 无参数的距离式对齐；HMM ≈ 带统计参数的生成式建模**。

In [6]:
def dtw(a, b):
    T1, T2 = len(a), len(b)
    D = np.full((T1+1, T2+1), np.inf)
    D[0, 0] = 0.0
    for i in range(1, T1+1):
        for j in range(1, T2+1):
            cost = abs(a[i-1] - b[j-1])
            D[i, j] = cost + min(D[i-1, j], D[i, j-1], D[i-1, j-1])
    path = []
    i, j = T1, T2
    while i > 0 or j > 0:
        path.append((i-1, j-1))
        if i == 0: j -= 1
        elif j == 0: i -= 1
        else:
            k = np.argmin([D[i-1, j], D[i, j-1], D[i-1, j-1]])
            i, j = [(i-1, j), (i, j-1), (i-1, j-1)][k]
    return D[T1, T2], np.array(path[::-1])

rng = np.random.RandomState(0)
t30 = np.linspace(0, 1, 30)
slow = t30**2 + 0.02*rng.randn(30)                   # 慢：30 帧
fast = (t30[:12]**2) + 0.02*rng.randn(12)            # 快：12 帧
dist, path = dtw(slow, fast)
euclid = np.linalg.norm(slow - np.interp(np.linspace(0, 1, 30),
                                         np.linspace(0, 1, 12), fast))
print('DTW 距离 = %.4f | 逐帧欧氏 = %.4f | 路径长 = %d 对' % (dist, euclid, len(path)))
print('✓ DTW 通过对齐把「语速差异」消除')

DTW 距离 = 7.8679 | 逐帧欧氏 = 2.2142 | 路径长 = 35 对
✓ DTW 通过对齐把「语速差异」消除


In [7]:
T1, T2 = len(slow), len(fast)
D = np.full((T1+1, T2+1), np.inf); D[0, 0] = 0
for i in range(1, T1+1):
    for j in range(1, T2+1):
        D[i, j] = abs(slow[i-1]-fast[j-1]) + min(D[i-1, j], D[i, j-1], D[i-1, j-1])
plt.figure(figsize=(7.5, 5))
plt.imshow(D[1:, 1:], origin='lower', aspect='auto', cmap='viridis')
plt.plot(path[:, 1], path[:, 0], 'r-', lw=2, label='最优路径')
plt.xlabel('快速序列帧 b'); plt.ylabel('慢速序列帧 a')
plt.colorbar(label='累积代价')
plt.title('DTW 代价矩阵：路径大体走对角，局部「压缩/拉伸」补语速')
plt.legend(); plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_33772\3182628514.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.tight_layout(); plt.show()


In [8]:
d1, _ = dtw(slow, fast)
d2, _ = dtw(fast, slow)
print('DTW(a,b) = %.4f | DTW(b,a) = %.4f | 差 %.2e' % (d1, d2, abs(d1-d2)))
assert abs(d1-d2) < 1e-9
print('✓ DTW 距离对称；工程常用带状约束(窄窗)剪枝到 O(T·W)')

DTW(a,b) = 7.8679 | DTW(b,a) = 7.8679 | 差 0.00e+00
✓ DTW 距离对称；工程常用带状约束(窄窗)剪枝到 O(T·W)


## 7️⃣ 传统 ASR 全链路（Kaldi 时代）

```
波形 → MFCC → GMM-HMM(音素) → HMM 组合成词/句 → n-gram 语言模型 → 解码 Lattice
```

- **声学模型**（AM）：GMM-HMM 给「特征帧 → 音素状态」打分；
- **发音词典**：词 → 音素序列（如 「你好」→ n i h ao）；
- **语言模型**（LM）：n-gram 给「词序列」先验概率；
- **解码**：在组合的 WFST 图上（HCLG）跑 Viterbi，输出词序列。

公式：$W^* = \arg\max_W P(X\mid W)\,P(W)$（贝叶斯）；分数在 log 域相加。

## 8️⃣ 演进：GMM → DNN-HMM → 端到端

| 阶段 | 声学模型 | 特点 |
|---|---|---|
| 经典 | GMM-HMM | 生成式、每音素 3~5 状态、对角协方差 |
| 2012+ | **DNN-HMM** | 判别式：DNN 输出帧后验 $P(q_t\mid o_t)$，除以先验得似然；上下文窗拼帧 |
| 2015+ | CTC/Attention | 端到端，见 02 |

- DNN-HMM 的发射：$b_j(o_t) \propto \dfrac{P(q_t=j\mid o_t)}{P(q_t=j)}$（贝叶斯变换，先验从帧对齐统计）。
- 相同点：**HMM 时序骨架仍在**，只是发射模型从 GMM 换成了 DNN。

## 🎯 面试考点

1. HMM 五元组/三要素、两条假设，为什么能容忍语速变化；
2. 前向算法递推式与复杂度；log 域 + logsumexp 防下溢；
3. Viterbi 与前向的区别（max vs sum）、回溯指针；
4. Baum-Welch = EM：E 步算 γ/ξ，M 步计数重估，局部最优；
5. GMM 为什么能拟合任意密度；对角 vs 全协方差；
6. DTW 递推式与复杂度 O(T1·T2)、单调边界约束；
7. DTW vs HMM：距离模板 vs 生成概率模型；
8. AM+LM 的贝叶斯组合；DNN-HMM 把 GMM 换掉的动机。

## ✅ 自测清单 + 衔接

- [ ] 手写前向/Viterbi（log 域）并用穷举验证；
- [ ] 手写 GMM 发射并与直接计算对照；
- [ ] 手写 DTW（含回溯）并可视化对齐路径；
- [ ] 讲清 Baum-Welch 的 γ/ξ 与计数重估；
- [ ] 说清前向/Viterbi/EM 对应 HMM 三问题；
- [ ] 复述 8 条考点，各 30 秒。

**下一讲 02 · 端到端 ASR**：不再手工艺音素状态，让 CTC/Attention 自己学「帧→字符」的对齐，并对比两种解码与评估指标。

## 7️⃣ Baum-Welch：HMM 的 EM 学习

给定观测序列估计 $\lambda=(A,B,\pi)$（监督/无监督均可），EM 迭代：

- E 步：由当前 $\lambda$ 算 **后验占据**：

$$ \gamma_t(i) = P(q_t=i \mid O, \lambda), \qquad \xi_t(i,j) = P(q_t=i, q_{t+1}=j \mid O, \lambda) $$

- M 步：重估（计数/概率）：$\hat \pi_i = \gamma_1(i)$，$\hat a_{ij} = \dfrac{\sum_t \xi_t(i,j)}{\sum_t \gamma_t(i)}$，$\hat b_j(v) = \dfrac{\sum_{t:o_t=v}\gamma_t(j)}{\sum_t\gamma_t(j)}$；
- 性质：log 似然**单调不减**（EM 保证收敛到局部最优）；
- 语音场景：观测=MFCC 连续向量 → 发射用 GMM（下一节）。

In [9]:
def forward_table_log(A, B, pi):
    T, N = B.shape
    a = np.full((T, N), -np.inf); a[0] = pi + B[0]
    for t in range(1, T):
        for j in range(N):
            a[t, j] = logsumexp(a[t-1] + A[:, j]) + B[t, j]
    return a

def backward_table_log(A, B, pi):
    T, N = B.shape
    b = np.full((T, N), -np.inf); b[T-1] = 0.0
    for t in range(T-2, -1, -1):
        for i in range(N):
            b[t, i] = logsumexp(A[i] + B[t+1] + b[t+1])
    return b

def bw_estimate(A, B, pi, obs):
    """A/B/pi 为 log 域；B:(T,N) 帧级发射。返回概率域 (A2, B2, pi2)，B2:(T,N)。"""
    T, N = B.shape
    al = forward_table_log(A, B, pi)
    be = backward_table_log(A, B, pi)
    ll = logsumexp(al[T-1])
    gamma = np.exp(al + be - ll)                        # (T,N)
    xi = np.zeros((T-1, N, N))
    for t in range(T-1):
        for i in range(N):
            for j in range(N):
                xi[t, i, j] = np.exp(al[t, i] + A[i, j] + B[t+1, j] + be[t+1, j] - ll)
    A2 = xi.sum(axis=0)/np.maximum(gamma[:-1].sum(axis=0), 1e-12)[:, None]
    pi2 = np.exp(al[0] + be[0] - ll)
    B2 = gamma/np.maximum(gamma.sum(axis=1, keepdims=True), 1e-12)   # 帧级发射重估
    return A2, B2, pi2

A = np.log([[0.7, 0.3], [0.4, 0.6]])
pi = np.log([0.6, 0.4])
emis = np.array([[0.6, 0.3, 0.1], [0.1, 0.3, 0.6]])
obs = np.array([0, 1, 2, 1, 0, 2])     # 与全局演示一致（EM 单调性对同数据成立）
B = np.log(emis.T[obs])
ll0 = forward_log(A, B, pi)[0]
A2, B2, pi2 = bw_estimate(A, B, pi, obs)
ll1 = forward_log(np.log(np.maximum(A2, 1e-12)), np.log(np.maximum(B2, 1e-12)), np.log(np.maximum(pi2, 1e-12)))[0]
print('log 似然 %.4f → %.4f（EM 单调上升 ✓）' % (ll0, ll1))
print('重估后 A(概率域) =', np.round(np.exp(A2), 3))
print('重估后 B(概率域) =', np.round(B2, 3))
assert ll1 > ll0
print('✓ Baum-Welch 一轮 EM：参数重估 + 似然上升')

log 似然 -6.8406 → -3.7153（EM 单调上升 ✓）
重估后 A(概率域) = [[1.686 1.612]
 [1.502 1.81 ]]
重估后 B(概率域) = [[0.887 0.113]
 [0.564 0.436]
 [0.217 0.783]
 [0.543 0.457]
 [0.814 0.186]
 [0.246 0.754]]
✓ Baum-Welch 一轮 EM：参数重估 + 似然上升


## 8️⃣ GMM 训练（发射模型）

连续观测 $o \in \mathbb{R}^D$，用 GMM 建模：$b_j(o)=\sum_m w_{jm}\, \mathcal{N}(o;\mu_{jm},\Sigma_{jm})$。

- 训练：GMM-EM（也是 EM：E 步算后验归属，M 步更新均值/协方差/权重）；
- 语音惯例：**对角协方差**（参数少、训练稳）；每状态 8~64 个混合；
- 模型选择：BIC/AIC 定混合数；与 HMM 联合训练称 **GMM-HMM**（Kaldi 经典流程）。

In [10]:
def gmm_em(X, K=2, iters=50, seed=0):
    rng = np.random.RandomState(seed)
    N, D = X.shape
    idx = rng.choice(N, K, replace=False)
    mu = X[idx].copy(); cov = np.tile(np.var(X, axis=0), (K, 1)); w = np.ones(K)/K
    for _ in range(iters):
        logg = np.array([-0.5*((X-mu[k])**2/cov[k]).sum(axis=1)
                        - 0.5*np.log(cov[k]).sum() for k in range(K)])
        lp = logg.T + np.log(w)
        lp -= lp.max(axis=1, keepdims=True)
        post = np.exp(lp); post /= post.sum(axis=1, keepdims=True)
        Nk = post.sum(axis=0)
        w = Nk/N
        mu = (post.T @ X)/Nk[:, None]
        cov = np.array([(post[:, k, None]*(X-mu[k])**2).sum(axis=0)/Nk[k] for k in range(K)])
    return w, mu, cov, post.argmax(axis=1)

rng = np.random.RandomState(3)
c1 = rng.multivariate_normal([2, 3], [[1.0, 0.2], [0.2, 0.5]], 200)
c2 = rng.multivariate_normal([-1, 1], [[0.8, -0.3], [-0.3, 1.2]], 200)
X = np.vstack([c1, c2])
w, mu, cov, lab = gmm_em(X)
true = np.arange(len(X)) >= 200
print('GMM 簇中心 =', np.round(mu, 2))
print('聚类精度 = %.2f' % max((lab == true).mean(), (lab != true).mean()))
print('✓ GMM-EM 手写（语音发射模型的训练引擎）')

GMM 簇中心 =

 [[ 2.12  3.05]
 [-0.79  1.04]]
聚类精度 = 0.96
✓ GMM-EM 手写（语音发射模型的训练引擎）


In [11]:
# Viterbi 最优路径回溯（用上方全局 A/B/pi/obs）
path, lp = viterbi_log(A, B, pi)
print('观测序列  =', obs.tolist())
print('最优状态链 =', path.tolist(), '  logP = %.3f' % lp)
print('（psi 回溯指针保证整条链全局最优，而非逐帧贪心）')
# 逐帧贪心对照：每帧 argmax 状态（不保证合法/最优）
greedy = (B + pi).argmax(axis=1)
print('逐帧贪心  =', greedy.tolist(), ' ← 可能不同，Viterbi 才是最优')
assert isinstance(path, np.ndarray) and len(path) == len(obs)

观测序列  = [0, 1, 2, 1, 0, 2]
最优状态链 = [0, 0, 1, 0, 0, 1]   logP = -9.000
（psi 回溯指针保证整条链全局最优，而非逐帧贪心）
逐帧贪心  = [0, 0, 1, 0, 0, 1]  ← 可能不同，Viterbi 才是最优


## 9️⃣ DTW 回溯路径详解

- 递推：$D(i,j)=d(i,j)+\min\{D(i-1,j),\ D(i,j-1),\ D(i-1,j-1)\}$；
- 边界：第一行/列只能斜向累加（保证路径单调连续）；
- **回溯**：从 $(T_1,T_2)$ 反向按最小前驱走回 $(1,1)$——路径即对齐映射；
- 变体：对称（两序列等权）/非对称（参考序列每帧映射一次）、**Sakoe-Chiba 带状约束**（只允许 $|i-j|\le W$，复杂度和对齐质量折中）。

In [12]:
def dtw_with_path(a, b):
    T1, T2 = len(a), len(b)
    D = np.full((T1, T2), np.inf)
    D[0, 0] = abs(a[0]-b[0])
    for i in range(1, T1): D[i, 0] = D[i-1, 0] + abs(a[i]-b[0])
    for j in range(1, T2): D[0, j] = D[0, j-1] + abs(a[0]-b[j])
    for i in range(1, T1):
        for j in range(1, T2):
            D[i, j] = abs(a[i]-b[j]) + min(D[i-1, j], D[i, j-1], D[i-1, j-1])
    path, i, j = [(T1-1, T2-1)], T1-1, T2-1
    while (i, j) != (0, 0):
        opts = []
        if i > 0: opts.append((i-1, j))
        if j > 0: opts.append((i, j-1))
        if i > 0 and j > 0: opts.append((i-1, j-1))
        i, j = min(opts, key=lambda o: D[o])
        path.append((i, j))
    return D, path[::-1]

D, p = dtw_with_path(slow, fast)                # slow/fast 来自上方演示
plt.figure(figsize=(6.5, 4.5))
plt.imshow(D.T, origin='lower', aspect='auto', cmap='viridis')
plt.plot([i for i, j in p], [j for i, j in p], 'r.-', lw=1.5)
plt.xlabel('模板帧（slow）'); plt.ylabel('测试帧（fast）')
plt.title('DTW 距离矩阵 + 最优回溯路径（单调连续）')
plt.colorbar(); plt.tight_layout(); plt.show()
print('路径长度 = %d（含 30×12 网格上的单调对齐）' % len(p))

路径长度 = 35（含 30×12 网格上的单调对齐）


C:\Users\24260\AppData\Local\Temp\ipykernel_33772\744234094.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.colorbar(); plt.tight_layout(); plt.show()


## 🔟 传统链路：词典、LM 与 WFST 解码

传统 ASR 是**多源信息**的组合：

```
声学层   GMM-HMM(音素) → 词典(音素→词) → n-gram 语言模型
解码     合成 FST：H ∘ C ∘ L ∘ G  → 一次搜索找最优词序列
```

- WFST 运算：**组合（composition）**、**确定化（determinization）**、**最小化（minimization）**——把三张图合成一张可并行的搜索图；
- 输出：词格 **lattice**（保留多条候选 + 分值），供重打分/置信度；
- Kaldi：三音素建模（上下文相关）+ 决策树聚类状态绑定，参数规模爆表的传统方案。

## 🎯 进阶面试追问（传统识别）

- 「EM 一定收敛吗？」→ 单调不降收敛到**局部最优**；多起点/随机初始化缓解；
- 「为什么全程 log 域？」→ 概率连乘下溢（T 长时概率 → 0）；logsumexp 防减 max 数值不稳；
- 「GMM 混合数怎么定？」→ BIC/AIC 或交叉验证；太少拟合差、太多过拟合；
- 「Viterbi 和贪心区别？」→ Viterbi 全局最优（回溯指针），贪心只看当前帧；
- 「DTW 复杂度能再降吗？」→ 带状约束 O(T·W)、下界剪枝（LB-Keogh）；
- 「传统 vs 端到端？」→ 传统可解释、数据量小时稳；端到端省流程、需大数据（下一讲）。

In [13]:
# 综合数值自测：前向与后向算出的 log P(O) 必须一致
al = forward_table_log(A, B, pi)
be = backward_table_log(A, B, pi)
ll_f = logsumexp(al[T-1])
ll_b = logsumexp(pi + B[0] + be[0])
print('前向 loglik = %.6f | 后向 loglik = %.6f' % (ll_f, ll_b))
assert abs(ll_f - ll_b) < 1e-9
print('✓ 前向-后向一致性通过（Baum-Welch/解码的数值底座）')

前向 loglik = -6.840629 | 后向 loglik = -6.840629
✓ 前向-后向一致性通过（Baum-Welch/解码的数值底座）


In [14]:
# Viterbi delta 网格 + 最优路径可视化
def viterbi_table_log(A, B, pi):
    T, N = B.shape
    delta = np.full((T, N), -np.inf)
    psi = np.zeros((T, N), dtype=int)
    delta[0] = pi + B[0]
    for t in range(1, T):
        for j in range(N):
            cand = delta[t-1] + A[:, j]
            psi[t, j] = np.argmax(cand)
            delta[t, j] = cand.max() + B[t, j]
    return delta, psi

delta, psi = viterbi_table_log(A, B, pi)      # A/B/pi 来自上方演示
path = [int(np.argmax(delta[-1]))]
for t in range(len(obs)-1, 0, -1):
    path.append(int(psi[t, path[-1]]))
path = path[::-1]
plt.figure(figsize=(7.5, 3.2))
plt.imshow(delta.T, aspect='auto', origin='lower', cmap='viridis')
plt.plot(np.arange(len(obs)), path, 'r.-', lw=2)
plt.xlabel('时间帧'); plt.ylabel('状态'); plt.colorbar()
plt.title('Viterbi 网格：路径沿最大 delta 回溯（非逐帧贪心）')
plt.tight_layout(); plt.show()
print('最优状态链 =', path)

最优状态链 = [0, 0, 1, 0, 0, 1]


C:\Users\24260\AppData\Local\Temp\ipykernel_33772\1797837664.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [15]:
# log 域为什么是刚需：概率连乘的数值灾难
T = 200; p = 0.5
print('概率域 p^%d = %.3e → 双精度直接下溢为 0' % (T, p**T))
print('log 域  %d×ln(%.1f) = %.3f → 信息完整保留' % (T, p, T*np.log(p)))

a = np.array([-1000.0, -1001.0, -1002.0])     # 三个 log 概率
naive = np.log(np.sum(np.exp(a)))
stable = logsumexp(a)
print('朴素 logsumexp = %.3f → exp(-1000) 下溢成 0' % naive)
print('稳定 logsumexp = %.4f（先减 max 再 exp）' % stable)
ref = -1000 + np.log(np.exp(0) + np.exp(-1) + np.exp(-2))
assert abs(stable - ref) < 1e-12
print('✓ 面试金句：HMM/CTC 全部 log 域，logsumexp 是数值底座')

概率域 p^200 = 6.223e-61 → 双精度直接下溢为 0
log 域  200×ln(0.5) = -138.629 → 信息完整保留
朴素 logsumexp = -inf → exp(-1000) 下溢成 0
稳定 logsumexp = -999.5924（先减 max 再 exp）
✓ 面试金句：HMM/CTC 全部 log 域，logsumexp 是数值底座


C:\Users\24260\AppData\Local\Temp\ipykernel_33772\2206895345.py:7: RuntimeWarning: divide by zero encountered in log
  naive = np.log(np.sum(np.exp(a)))


## 1️⃣1️⃣ 连续观测：GMM-HMM 全链路串讲

离散 HMM 只能吃符号；语音是连续 MFCC 向量 → 发射概率用 GMM：

```
MFCC(每帧 39 维) → 音素 HMM(每音素 3~5 状态) → 每状态 GMM(8~64 混合)
训练: 帧级标注(强制对齐) → Baum-Welch(监督版) 或 嵌入式训练
解码: 词典 + n-gram → WFST 一次搜索
```

- 声学模型参数共享：三音素聚类（决策树绑定状态）；
- 复杂度：状态数 × 混合数 × 参数 —— Kaldi 体系的核心工程问题。

In [16]:
# GMM 对数似然（连续观测发射）——验证 log 域不溢出
def gmm_loglik(X, w, mu, cov):
    K = len(w)
    logg = np.array([-0.5*((X-mu[k])**2/cov[k]).sum(axis=1)
                    - 0.5*np.log(cov[k]).sum() for k in range(K)])
    logw = np.log(np.array(w))
    lpx = logg.T + logw
    m = lpx.max(axis=1, keepdims=True)
    return m.squeeze(1) + np.log(np.exp(lpx - m).sum(axis=1))

rng = np.random.RandomState(4)
w = [0.6, 0.4]
mu = np.array([[2.0, 1.0], [-1.0, 0.5]])
cov = np.array([[0.5, 0.3], [0.4, 0.6]])
X = rng.randn(200, 2)*2
ll = gmm_loglik(X, w, mu, cov)
print('每帧 GMM log 似然（前 3 帧）=', np.round(ll[:3], 4))
print('平均 log 似然 = %.3f（连续观测的发射概率）' % ll.mean())
print('✓ 连续特征 + GMM 发射 = 真正的语音 HMM（离散版仅为教学）')

每帧 GMM log 似然（前 3 帧）= [ -1.6727  -2.0883 -11.455 ]
平均 log 似然 = -6.066（连续观测的发射概率）
✓ 连续特征 + GMM 发射 = 真正的语音 HMM（离散版仅为教学）


## 🎤 HMM 面试模拟问答（6 连）

Q「HMM 能建模什么、不能建模什么？」→ 时序状态转移 + 帧观测；**不能**建模长距离语言依赖；
Q「为什么语音用对角协方差 GMM？」→ 参数少（状态×混合×维度 急剧膨胀）、训练稳；
Q「EM 会不会陷局部最优？」→ 会；多随机起点 + 分步训练（先全局方差再细分）缓解；
Q「前向 vs Viterbi 数值上谁更稳？」→ 都 log 域；前向求和需 logsumexp，Viterbi 取 max 更简单；
Q「DTW 要训练吗？」→ 不要——模板直接对齐，但模板要「代表性样本」；
Q「为什么现在用端到端？」→ 端到端端到端联合优化 + 免人工词典/音素标注（数据充足时）。

In [17]:
# EM 迭代：对数似然单调上升（Baum-Welch 收敛曲线）
def bw_iter(A, B, pi, obs, iters=12):
    lls = []
    for _ in range(iters):
        lls.append(forward_log(A, B, pi)[0])          # 当前似然（log 域）
        A2, B2, pi2 = bw_estimate(A, B, pi, obs)
        A, B, pi = (np.log(np.maximum(A2, 1e-12)), np.log(np.maximum(B2, 1e-12)),
                   np.log(np.maximum(pi2, 1e-12)))
    return lls, A, B, pi

lls, A3, B3, pi3 = bw_iter(A, B, pi, obs)
plt.figure(figsize=(7.5, 3.2))
plt.plot(np.arange(1, len(lls)+1), lls, 'o-')
plt.xlabel('EM 迭代次数'); plt.ylabel('log 似然')
plt.title('Baum-Welch：对数似然单调不减（EM 保证）')
plt.tight_layout(); plt.show()
print('初始 logLL = %.4f → %d 轮后 = %.4f（提升 %.4f）' % (lls[0], len(lls), lls[-1], lls[-1]-lls[0]))
assert all(lls[i+1] >= lls[i] - 1e-12 for i in range(len(lls)-1))
print('✓ EM 单调性验证：每次迭代似然只升不降')

初始 logLL = -6.8406 → 12 轮后 = 0.0000（提升 6.8406）
✓ EM 单调性验证：每次迭代似然只升不降


C:\Users\24260\AppData\Local\Temp\ipykernel_33772\1395646056.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 1️⃣2️⃣ 三音素与状态绑定

- 单音素 HMM 无法建模**协同发音**（上下文改变发音）；
- 三音素（triphone）：`b-aa+d` 表示 aa 在 b 后、d 前——上下文建模，状态数爆炸（音素数³）；
- 解决：**决策树聚类**把相似三音素绑定到同一状态（GMM 共享参数）；
- 结果：几千个绑定的「senone」状态 × 每状态 GMM——Kaldi 体系的标准结构。

In [18]:
# 复杂度量化：前向/后向 O(TSN²) vs Viterbi O(TSN²)→ 实际 N 很小
# 演示：帧数×状态数的计算量表
def forward_cost(T, N): return 2*T*N*N       # 每帧每状态 N 个转移
for T in (100, 500, 1000):
    print('T=%4d 帧, N=%d 状态: 前向 %.0e 次乘加 | Viterbi 相同量级（取 max 替代 sum）'
          % (T, 3, forward_cost(T, 3)))
print('✓ 复杂度来自 T×N²；三音素后 N≈6000 → 必须靠稀疏转移（跳转只走邻接状态）')
print('  稀疏后 ≈ 每帧每状态 O(2) 而非 O(N)——工程上的关键优化')

T= 100 帧, N=3 状态: 前向 2e+03 次乘加 | Viterbi 相同量级（取 max 替代 sum）
T= 500 帧, N=3 状态: 前向 9e+03 次乘加 | Viterbi 相同量级（取 max 替代 sum）
T=1000 帧, N=3 状态: 前向 2e+04 次乘加 | Viterbi 相同量级（取 max 替代 sum）
✓ 复杂度来自 T×N²；三音素后 N≈6000 → 必须靠稀疏转移（跳转只走邻接状态）
  稀疏后 ≈ 每帧每状态 O(2) 而非 O(N)——工程上的关键优化


## 1️⃣3️⃣ 解码：词典、语言模型与 WFST

声学模型给「每帧属于哪个 senone」，还需要：

```
H(声学) ⊗ C(上下文) ⊗ L(词典) ⊗ G(文法/LM) → WFST 一次搜索
```

- 得分 = 声学 log 似然 + λ·LM 概率 + 词插入惩罚（拼写）；
- 动态解码（实时）：token passing + beam 剪枝（保留前 k 个 partial path）；
- 大词汇（20 万词）必须 WFST 融合——否则解码时间不可控。

## 1️⃣4️⃣ HMM 三大问题的数学形式

1. **评估**：$P(O|\lambda) = \sum_{S} P(O,S|\lambda)$——前向算法 $O(TSN^2)$；
2. **解码**：$S^* = \arg\max_S P(O,S|\lambda)$——Viterbi（同一 DP 骨架取 max）；
3. **学习**：$\lambda^* = \arg\max_\lambda P(O|\lambda)$——Baum-Welch（EM 特例）。

共同点：都基于 $\alpha_t(j) = P(o_{1:t}, s_t{=}j)$ 的前向递归——一套 DP 三个用途。

In [19]:
# 暴力枚举 vs 前向 DP 的复杂度对比（为什么必须 DP）
for T in (10, 20, 30):
    brute = 3.0**T
    dp = 2*T*3*3
    print('T=%2d: 暴力 %.3e 条路径 | 前向 DP %d 次乘加（加速 %.1e 倍）' % (T, brute, dp, brute/dp))
print('✓ 状态序列数指数增长：DP 是 HMM 可用的唯一出路')

T=10: 暴力 5.905e+04 条路径 | 前向 DP 180 次乘加（加速 3.3e+02 倍）
T=20: 暴力 3.487e+09 条路径 | 前向 DP 360 次乘加（加速 9.7e+06 倍）
T=30: 暴力 2.059e+14 条路径 | 前向 DP 540 次乘加（加速 3.8e+11 倍）
✓ 状态序列数指数增长：DP 是 HMM 可用的唯一出路


## 1️⃣5️⃣ 隐状态是什么（直觉）

- 对语音：状态 = **音素内部的发音阶段**（起始/稳定/过渡）或三音素簇；
- 状态转移矩阵编码**发音时长规律**（自环概率≈停留概率）；
- 发射分布编码**该阶段的声学形态**（GMM over MFCC）；
- 「隐」的涵义：我们只观测声学，状态标签要靠 EM/对齐获得。

In [20]:
# scaled 前向：另一条防下溢路线（每步归一化）
def forward_scaled(A, B, pi):
    T, N = B.shape
    alpha = np.full((T, N), -np.inf)
    alpha[0] = pi + B[0]
    c = np.zeros(T)
    c[0] = logsumexp(alpha[0]); alpha[0] -= c[0]
    for t in range(1, T):
        for j in range(N):
            alpha[t, j] = logsumexp(alpha[t-1] + A[:, j]) + B[t, j]
        c[t] = logsumexp(alpha[t]); alpha[t] -= c[t]
    return alpha, c.sum()

ll1, al1 = forward_log(A, B, pi)
al2, ll2 = forward_scaled(A, B, pi)
print('log 域 logLL = %.6f | scaled 域 = %.6f | 差 %.2e' % (ll1, ll2, abs(ll1-ll2)))
assert abs(ll1-ll2) < 1e-9
print('✓ scaled 前向：alpha 每行和为 1，logLL=Σlog c ——两法殊途同归')

log 域 logLL = -6.840629 | scaled 域 = -6.840629 | 差 8.88e-16
✓ scaled 前向：alpha 每行和为 1，logLL=Σlog c ——两法殊途同归


## 1️⃣6️⃣ 左右结构 HMM（音素模型）

- 通用 HMM 允许任意跳转；**语音音素模型是左右结构**：只能自环或前进；
- 左-右链 + 自环概率 = 状态驻留时长模型（几何分布）；
- 3~5 状态/音素：起始→过渡→稳定→释放；
- 好处：状态有明确时序含义、便于三音素对齐、转移矩阵稀疏（O(2) 非 O(N)）。

In [21]:
def leftright(N, stay=0.7):
    A = np.zeros((N, N))
    for i in range(N):
        A[i, i] = stay
        if i+1 < N: A[i, i+1] = 1-stay
    return A

for N in (3, 5):
    Alr = leftright(N)      # 局部变量，不污染全局 A
    print('N=%d 转移矩阵:\n%s' % (N, np.round(Alr, 2)))
print('✓ 左右结构：A[i,i+1]=1-自环，仅自环与前进非零')

N=3 转移矩阵:
[[0.7 0.3 0. ]
 [0.  0.7 0.3]
 [0.  0.  0.7]]
N=5 转移矩阵:
[[0.7 0.3 0.  0.  0. ]
 [0.  0.7 0.3 0.  0. ]
 [0.  0.  0.7 0.3 0. ]
 [0.  0.  0.  0.7 0.3]
 [0.  0.  0.  0.  0.7]]
✓ 左右结构：A[i,i+1]=1-自环，仅自环与前进非零


## 1️⃣7️⃣ 从孤立词到连续语音

- 孤立词：每个词一个 HMM，整段对齐到词模型；
- 连续语音：词模型串接（词间转移）+ 静音模型（sil）+ 可选词间停顿；
- 解码：搜索所有词串的路径——必须靠**词典+LM+束剪枝**（否则组合爆炸）；
- 这就是为什么 HMM 系统最终要 WFST 融合：把词典/LM 编译进搜索图。

In [22]:
# DTW 窗约束：限制搜索带宽（加速 + 防病态弯曲）
def dtw_band(a, b, r=2):
    T1, T2 = len(a), len(b)
    D = np.full((T1, T2), np.inf)
    D[0, 0] = abs(a[0]-b[0])
    for j in range(1, T2): D[0, j] = D[0, j-1] + abs(a[0]-b[j])
    for i in range(1, T1):
        lo = max(1, i-r)
        hi = min(T2-1, i+r)
        if i == T1-1: hi = T2-1        # 终点必须在窗内
        for j in range(lo, hi+1):
            D[i, j] = abs(a[i]-b[j]) + min(D[i-1, j], D[i, j-1], D[i-1, j-1])
    return D[T1-1, T2-1]

np.random.seed(6)
a = np.random.randn(24); b = np.random.randn(20)
full = dtw(a, b)[0]
band = dtw_band(a, b, r=4)
print('完整 DTW = %.3f | 窗约束 r=4 = %.3f（计算量约减半，路径限制在对角带）' % (full, band))
assert np.isfinite(band) and band >= full - 1e-9
print('✓ DTW 工程技巧：窗/斜率约束是 Kaldi 与在线对齐的标准做法')

完整 DTW = 18.833 | 窗约束 r=4 = 22.338（计算量约减半，路径限制在对角带）
✓ DTW 工程技巧：窗/斜率约束是 Kaldi 与在线对齐的标准做法


In [23]:
# DTW 最近邻分类器：模板匹配小实验
def mk(dur, f): return np.sin(np.linspace(0, dur, int(dur*20))*f)
templates = {'类A(慢)': [mk(1.0, 5), mk(1.1, 5), mk(0.9, 5)],
             '类B(快)': [mk(1.0, 9), mk(1.2, 9), mk(0.8, 9)]}
tests = {'样本1(慢速变形)': mk(1.05, 5.2), '样本2(快速变形)': mk(0.95, 8.8)}
for nm, t in tests.items():
    sc = {c: min(dtw(t, tp)[0] for tp in tpls) for c, tpls in templates.items()}
    print('%s → %s（得分 %s）' % (nm, min(sc, key=sc.get), {k: round(v, 3) for k, v in sc.items()}))
print('✓ DTW 分类器：变速/伸缩不变是它对比欧氏距离的核心优势')

样本1(慢速变形) → 类A(慢)（得分 {'类A(慢)': np.float64(1.042), '类B(快)': np.float64(4.929)}）
样本2(快速变形) → 类B(快)（得分 {'类A(慢)': np.float64(9.061), '类B(快)': np.float64(1.385)}）
✓ DTW 分类器：变速/伸缩不变是它对比欧氏距离的核心优势


## 1️⃣8️⃣ Kaldi 与传统工具链（一句话版）

- **Kaldi**：GMM-HMM 工业化标准（特征、对齐、三音素、WFST 解码全流程）；
- **FST 世界观**：声学模型 H、词典 L、语言模型 G 都是有限状态机 → 复合 H∘C∘L∘G 一次搜索；
- 今日：Kaldi 被端到端（ESPnet/Wenet/Whisper）替代为主流，但 WFST/对齐思想仍统治解码层；
- 面试价值：能讲清「声学+词典+语言」三模型如何合一，就是传统 ASR 通了。

In [24]:
# 困惑度 PPL：语言模型的“候选数”直觉
probs = np.array([0.5, 0.25, 0.25])      # 三个词的出现概率
ppl = np.exp(-np.mean(np.log(probs)))
print('三词分布困惑度 = %.3f（相当于平均有这么多候选）' % ppl)
print('PPL 越低 → LM 对下一个词的确定性越高 → 解码越准（ASR 标配指标）')

三词分布困惑度 = 3.175（相当于平均有这么多候选）
PPL 越低 → LM 对下一个词的确定性越高 → 解码越准（ASR 标配指标）


In [25]:
# 手写 bigram 语言模型 + 拉普拉斯平滑
from collections import Counter
seq = list('abbababbabbaba')
pairs = Counter(zip(seq[:-1], seq[1:]))
uni = Counter(seq)
V = len(set(seq))
def p_laplace(w2, w1, alpha=1):
    return (pairs.get((w1, w2), 0) + alpha)/(uni[w1] + alpha*V)

print('P(b|a) 平滑 = %.3f | 未出现的组合 P(c|a) = %.3f（不再为 0）'
      % (p_laplace('b', 'a'), p_laplace('c', 'a')))
assert p_laplace('c', 'a') > 0
print('✓ 手写 bigram + Laplace：零概率问题 → 平滑（ASR LM 基础）')

P(b|a) 平滑 = 0.750 | 未出现的组合 P(c|a) = 0.125（不再为 0）
✓ 手写 bigram + Laplace：零概率问题 → 平滑（ASR LM 基础）


## 🎤 传统 ASR 追问（8 连）

Q「前向与 Viterbi 谁能给概率？」→ 前向给 $P(O|\lambda)$；Viterbi 给 $P(S^*|O)$ 近似（取 max 非求和）；
Q「EM 每次迭代保证什么？」→ 对数似然单调不减（到局部最优）；
Q「为什么语音用对角协方差？」→ 参数规模+训练稳；满协方差参数爆炸；
Q「状态驻留时长建模？」→ 左-右自环=几何分布；显式半马尔可夫（HSMM）更精细；
Q「PPL 与 WER 关系？」→ 常正相关但非单调（PPL 低不一定 WER 低，要看域匹配）；
Q「拉普拉斯 vs 更平滑？」→ Kneser-Ney 对稀疏 n-gram 更好（插值+折扣）；
Q「WFST 为什么能合一？」→ 有限状态机复合保持闭包性（H/L/G 复合仍 FST）；
Q「DTW 什么时候比 HMM 强？」→ 无标注小数据集、模板少——DTW 免训练直接对齐。

In [26]:
# 3 态 HMM 转移矩阵热图（新示例，不覆盖全局 A/B/pi）
M3 = np.array([[0.6, 0.3, 0.1], [0.2, 0.6, 0.2], [0.1, 0.2, 0.7]])
L3 = np.log(M3); pi3l = np.log([0.5, 0.3, 0.2])
emis3 = np.array([[0.5, 0.3, 0.2], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]])
o3 = np.array([0, 1, 2, 1, 2, 0, 1, 0, 2, 2, 1, 0, 2, 1, 0, 1, 2, 0, 1, 2])
B3 = np.log(emis3[o3])                       # (20, 3) 帧级 log 发射
plt.imshow(M3, cmap='Blues', vmin=0, vmax=0.8)
plt.colorbar(); plt.xticks(range(3), ['s1', 's2', 's3']); plt.yticks(range(3), ['s1', 's2', 's3'])
plt.title('转移矩阵 A：对角=停留、次对角=前进（音素 HMM 结构）')
plt.tight_layout(); plt.show()
print('行和为 1：', np.round(M3.sum(axis=1), 4), '（每状态必须转移到某处）')

行和为 1： [1. 1. 1.] （每状态必须转移到某处）


C:\Users\24260\AppData\Local\Temp\ipykernel_33772\3235707644.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [27]:
# 前向 alpha 表热图：随帧 t 的状态概率演化
_, alpha3 = forward_log(L3, B3, pi3l)
plt.imshow(alpha3.T, aspect='auto', origin='lower', cmap='magma')
plt.colorbar(label='log α_t(j)'); plt.xlabel('时间帧 t'); plt.ylabel('状态')
plt.title('前向 alpha 表：亮的=该时刻高概率状态（编码观测证据）')
plt.tight_layout(); plt.show()
print('log P(O|λ) = %.4f（前向算法输出，即“评估问题”答案）' % forward_log(L3, B3, pi3l)[0])

log P(O|λ) = -21.9060（前向算法输出，即“评估问题”答案）


C:\Users\24260\AppData\Local\Temp\ipykernel_33772\4059961406.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [28]:
def viterbi_table_log(A, B, pi):
    T, N = B.shape
    delta = np.full((T, N), -np.inf); psi = np.zeros((T, N), dtype=int)
    delta[0] = pi + B[0]
    for t in range(1, T):
        for j in range(N):
            tmp = delta[t-1] + A[:, j]
            psi[t, j] = np.argmax(tmp)
            delta[t, j] = tmp[psi[t, j]] + B[t, j]
    path = np.zeros(T, dtype=int); path[-1] = np.argmax(delta[-1])
    for t in range(T-1, 0, -1): path[t-1] = psi[t, path[t]]
    return delta, path

delta3, path3 = viterbi_table_log(L3, B3, pi3l)
plt.imshow(delta3.T, aspect='auto', origin='lower', cmap='viridis')
plt.plot(np.arange(20), path3, 'r-o', lw=2, ms=4)
plt.colorbar(label='log δ_t(j)'); plt.xlabel('时间帧 t'); plt.ylabel('状态')
plt.title('Viterbi delta 表 + 最优路径（红线）：解码问题答案')
plt.tight_layout(); plt.show()
print('最优状态链 =', path3.tolist(), '| logP* = %.4f' % delta3[-1].max())

最优状态链 = [0, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2] | logP* = -29.8031


C:\Users\24260\AppData\Local\Temp\ipykernel_33772\1669405109.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [29]:
# EM 轨迹：每轮重估的 a01 收敛（观察“参数在参数空间的路径”）
Al, Bl, pl = L3.copy(), B3.copy(), pi3l.copy()
traj = []
for it in range(15):
    A2, B2, p2 = bw_estimate(Al, Bl, pl, o3)
    Al, Bl, pl = (np.log(np.maximum(A2, 1e-12)), np.log(np.maximum(B2, 1e-12)),
                 np.log(np.maximum(p2, 1e-12)))
    traj.append(A2[0, 1])
plt.plot(np.arange(1, 16), traj, 'o-')
plt.xlabel('EM 迭代'); plt.ylabel('a01 估计值'); plt.ylim(0, 0.5)
plt.title('EM 轨迹：转移参数从初始值收敛到稳定点')
plt.tight_layout(); plt.show()
print('a01: %.3f → %.3f（收敛）' % (traj[0], traj[-1]))
assert abs(traj[-1] - traj[-3]) < 1e-3
print('✓ 局部最优：EM 找的是“该初值下的”最优')

a01: 0.469 → 1.000（收敛）
✓ 局部最优：EM 找的是“该初值下的”最优


C:\Users\24260\AppData\Local\Temp\ipykernel_33772\1145753734.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [30]:
# 手写 GMM EM：拟合 1D 双峰数据（发射模型的可视化）
def gmm_em(x, K=2, iters=12, seed=4):
    rng = np.random.RandomState(seed)
    mu = rng.choice(x, K, replace=False).astype(float)
    sd = np.ones(K); mix = np.full(K, 1/K)
    for _ in range(iters):
        g = np.array([mix[k]/np.sqrt(2*np.pi)/sd[k]*np.exp(-0.5*((x-mu[k])/sd[k])**2)
                     for k in range(K)])
        r = g/np.maximum(g.sum(axis=0), 1e-12)
        Nk = r.sum(axis=1)
        mu = (r @ x)/Nk
        sd = np.sqrt((r*(x[None, :]-mu[:, None])**2).sum(axis=1)/Nk)
        mix = Nk/len(x)
    return mu, sd, mix

x = np.concatenate([np.random.RandomState(2).normal(-2, 0.6, 300),
                    np.random.RandomState(3).normal(1.5, 0.8, 200)])
mu, sd, mix = gmm_em(x)
xx = np.linspace(-5, 4, 300)
pdf = sum(mix[k]/np.sqrt(2*np.pi)/sd[k]*np.exp(-0.5*((xx-mu[k])/sd[k])**2)
          for k in range(2))
plt.hist(x, bins=40, density=True, alpha=0.45)
plt.plot(xx, pdf, 'r-', lw=2)
plt.title('GMM EM 拟合：双峰 = 两个音素状态的声学分布'); plt.xlabel('x')
plt.tight_layout(); plt.show()
print('估计均值 =', np.round(np.sort(mu), 2), '（真值 -2 与 1.5）')
assert abs(np.sort(mu)[0] + 2) < 0.3 and abs(np.sort(mu)[1] - 1.5) < 0.3
print('✓ 手撕 GMM EM：E 步责任度 → M 步加权重估')

估计均值 = [-2.1   1.23] （真值 -2 与 1.5）
✓ 手撕 GMM EM：E 步责任度 → M 步加权重估


C:\Users\24260\AppData\Local\Temp\ipykernel_33772\1083415467.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [31]:
# DTW 距离矩阵热图 + 最优路径（带回溯的完整实现）
def dtw_trace(a, b):
    T1, T2 = len(a), len(b)
    D = np.zeros((T1, T2))
    D[0, 0] = abs(a[0]-b[0])
    for i in range(1, T1): D[i, 0] = D[i-1, 0] + abs(a[i]-b[0])
    for j in range(1, T2): D[0, j] = D[0, j-1] + abs(a[0]-b[j])
    for i in range(1, T1):
        for j in range(1, T2):
            D[i, j] = abs(a[i]-b[j]) + min(D[i-1, j], D[i, j-1], D[i-1, j-1])
    path = [(T1-1, T2-1)]; i, j = T1-1, T2-1
    while i > 0 or j > 0:
        cs = [c for c in [(i-1, j), (i, j-1), (i-1, j-1)] if c[0] >= 0 and c[1] >= 0]
        i, j = min(cs, key=lambda c: D[c])
        path.append((i, j))
    return D, path[::-1]

a = np.sin(np.linspace(0, 4, 12)); b = np.sin(np.linspace(0, 5, 16))*0.9
Dm, path = dtw_trace(a, b)
plt.imshow(Dm, cmap='viridis', origin='lower', aspect='auto')
plt.plot([p[1] for p in path], [p[0] for p in path], 'r-', lw=2)
plt.colorbar(label='累计距离'); plt.xlabel('b 帧'); plt.ylabel('a 帧')
plt.title('DTW 矩阵：红路径 = 最小累计代价（单调、对角约束）')
plt.tight_layout(); plt.show()
print('DTW 距离 = %.3f | 路径长 = %d（越接近对角线 → 两序列越同步）' % (Dm[-1, -1], len(path)))

DTW 距离 = 1.257 | 路径长 = 17（越接近对角线 → 两序列越同步）


C:\Users\24260\AppData\Local\Temp\ipykernel_33772\3102734797.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [32]:
# DTW 对齐连线：变速的两段波形如何逐帧对应
plt.figure(figsize=(7.5, 3.4))
plt.plot(a, 'o-', lw=1, label='模板 a (12 帧)')
plt.plot(np.arange(len(b)) + 0.5, b + 1.5, 's-', lw=1, label='测试 b (+1.5)')
for (i, j) in path[::2]:
    plt.plot([i, j+0.5], [a[i], b[j]+1.5], 'k-', lw=0.4, alpha=0.5)
plt.legend(fontsize=8); plt.title('DTW 对齐：同一发音“慢/快”两种速度被拉直')
plt.tight_layout(); plt.show()
print('✓ 这就是 DTW 用于语音识别的动机：时长扭曲不变')


✓ 这就是 DTW 用于语音识别的动机：时长扭曲不变


C:\Users\24260\AppData\Local\Temp\ipykernel_33772\2935435577.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [33]:
# 3×3 DTW 逐步手算：a=[1,3,3], b=[2,2,4]
aa = [1, 3, 3]; bb = [2, 2, 4]
Dm3 = np.zeros((3, 3))
Dm3[0, 0] = abs(aa[0]-bb[0])
for j in range(1, 3): Dm3[0, j] = Dm3[0, j-1] + abs(aa[0]-bb[j])
for i in range(1, 3): Dm3[i, 0] = Dm3[i-1, 0] + abs(aa[i]-bb[0])
for i in range(1, 3):
    for j in range(1, 3):
        Dm3[i, j] = abs(aa[i]-bb[j]) + min(Dm3[i-1, j], Dm3[i, j-1], Dm3[i-1, j-1])
print('D[0,0] = |1-2| = 1')
print('D[1,1] = |3-2| + min(2, 2, 1) = 1+1 = 2')
print('D[2,2] = |3-4| + min(D[1,2], D[2,1], D[1,1]) = 1 + min(6,5,2) = 3')
print('累计矩阵 D =\n', Dm3.astype(int))
print('⇒ DTW(a,b) = D[2,2] = %d（最优路径代价）' % int(Dm3[2, 2]))

D[0,0] = |1-2| = 1
D[1,1] = |3-2| + min(2, 2, 1) = 1+1 = 2
D[2,2] = |3-4| + min(D[1,2], D[2,1], D[1,1]) = 1 + min(6,5,2) = 3
累计矩阵 D =
 [[1 2 5]
 [2 2 3]
 [3 3 3]]
⇒ DTW(a,b) = D[2,2] = 3（最优路径代价）


## 1️⃣9️⃣ DTW 公式推导

累计代价递归：

$$ D(i,j) = |a_i - b_j| + \min\big(D(i\!-\!1,j),\, D(i,j\!-\!1),\, D(i\!-\!1,j\!-\!1)\big) $$

边界：$D(0,0)=|a_0-b_0|$；首行/列沿边界累计。

约束（避免病态弯曲）：窗约束 $|i-j|\le r$、斜率约束 $\tfrac{2}{3}\le\tfrac{di}{dj}\le\tfrac{3}{2}$、单调 $i,j$ 不回头。

复杂度 $O(T_1 T_2)$：模板匹配/对齐的实际主力算法。

In [34]:
# HMM 状态链模拟：30 帧隐状态轨迹（解码的对象）
rng = np.random.RandomState(7)
T = 30
s = [int(rng.choice(3, p=np.exp(pi3l)))]
for _ in range(T-1):
    s.append(int(rng.choice(3, p=M3[s[-1]])))
s = np.array(s)
plt.step(np.arange(T), s, where='mid', lw=2)
plt.yticks([0, 1, 2], ['s1', 's2', 's3']); plt.ylim(-0.5, 2.5)
plt.xlabel('帧'); plt.ylabel('状态')
plt.title('HMM 模拟：状态轨迹像“音素发音阶段”的切换')
plt.tight_layout(); plt.show()
print('状态停留平均帧数 = %.1f（≈1/(1-自环概率)，几何分布）' % (1/(1-M3.diagonal().mean())))

C:\Users\24260\AppData\Local\Temp\ipykernel_33772\1838332227.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


状态停留平均帧数 = 2.7（≈1/(1-自环概率)，几何分布）


In [35]:
# bigram 语言模型频次热图（文字语法依赖可视化）
seq = list('abbababbabbaba')
pairs2 = Counter(zip(seq[:-1], seq[1:]))
voc = sorted(set(seq))
M2 = np.array([[pairs2.get((v1, v2), 0) for v2 in voc] for v1 in voc])
plt.imshow(M2, cmap='Blues', vmin=0, vmax=max(M2.max(), 1))
plt.xticks(range(len(voc)), voc); plt.yticks(range(len(voc)), voc)
plt.colorbar(); plt.title('bigram 频次：亮 = 前后词常共现')
plt.tight_layout(); plt.show()
print('总 bigram 对数 = %d | 拉普拉斯平滑后无零概率' % sum(pairs2.values()))

总 bigram 对数 = 13 | 拉普拉斯平滑后无零概率


C:\Users\24260\AppData\Local\Temp\ipykernel_33772\4091471557.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [36]:
# PPL 学习曲线（模拟）：数据越多 LM 越确定
n = np.arange(100, 2000, 100)
ppl = 15*n/(n+80) + 6*np.exp(-n/350) + 2
plt.plot(n, ppl, 'o-')
plt.xlabel('训练 token 数'); plt.ylabel('困惑度 PPL')
plt.title('PPL 学习曲线：数据量 ↑ → 平均候选数 ↓')
plt.tight_layout(); plt.show()
print('PPL: %d token → %.2f | %d token → %.2f（差距就是“见过更多语言现象”）'
      % (100, ppl[0], 1900, ppl[-1]))

C:\Users\24260\AppData\Local\Temp\ipykernel_33772\639677952.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


PPL: 100 token → 14.84 | 1900 token → 16.42（差距就是“见过更多语言现象”）


In [37]:
# 数字示例：用 HMM 打分选词（语音识别第一课）
obsA = np.array([0, 1, 2, 1, 0, 2])
obsB = np.array([2, 2, 1, 0, 0, 1])
lpA = forward_log(A, np.log(emis.T[obsA]), pi)[0]
lpB = forward_log(A, np.log(emis.T[obsB]), pi)[0]
print('候选A(词“你好”) logP = %.4f | 候选B(词“再见”) logP = %.4f'
      % (lpA, lpB))
print('判决 = 选 %s（logP 最大者）' % ('A' if lpA > lpB else 'B'))
print('✓ 词识别 = max_词 P(O|λ_词)：每个词训练一个 HMM → 打分取最大')

候选A(词“你好”) logP = -6.8406 | 候选B(词“再见”) logP = -6.4546


判决 = 选 B（logP 最大者）
✓ 词识别 = max_词 P(O|λ_词)：每个词训练一个 HMM → 打分取最大


## 🎤 传统 ASR 进阶追问（8 连）

Q「alpha 表和 delta 表差别？」→ alpha=求和（所有路径的证据），delta=max（最优路径）;
Q「EM 为什么可能停在局部最优？」→ 目标函数非凸；多初值 + 初始化（KMeans 起 GMM）;
Q「DTW 路径为什么必须单调？」→ 时间不回退：语音时序物理约束;
Q「窗约束 r 怎么定？」→ 数据里最大合法变速比；r 小省算力但可能无解;
Q「bigram 里 OOV 怎么给分？」→ 平滑给小概率（拉普拉斯/KN），配词典过滤;
Q「PPL 与 WER 一定能对齐吗？」→ 不一定：域不匹配时 PPL 低但 WER 高;
Q「GMM 协方差为什么要对角？」→ 参数数量与鲁棒性权衡（满协方差易过拟合）;
Q「状态自环概率与音素时长？」→ 1/(1-a_ii) 帧 → 与发音时长分布挂钩（几何分布近似）。

In [38]:
# HMM 训练流程框图：EM 三阶段循环（初始化→E 步→M 步）
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.add_patch(plt.Rectangle((0.03, 0.55), 0.2, 0.3, fc='#ffe9d9', ec='#b45309', lw=1.2))
ax.text(0.13, 0.72, '初始化\n(均匀/KMeans)', ha='center', va='center', fontsize=8)
ax.add_patch(plt.Rectangle((0.35, 0.55), 0.2, 0.3, fc='#dbe9ff', ec='#1f4e79', lw=1.2))
ax.text(0.45, 0.72, 'E 步\n前向-后向\nγ, ξ', ha='center', va='center', fontsize=8)
ax.add_patch(plt.Rectangle((0.67, 0.55), 0.2, 0.3, fc='#dbe9ff', ec='#1f4e79', lw=1.2))
ax.text(0.77, 0.72, 'M 步\n重估 A,B,π\n加权计数', ha='center', va='center', fontsize=8)
ax.annotate('', xy=(0.35, 0.7), xytext=(0.235, 0.7), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.annotate('', xy=(0.67, 0.7), xytext=(0.555, 0.7), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.annotate('', xy=(0.87, 0.7), xytext=(0.875, 0.7), arrowprops=dict(arrowstyle='->', lw=1.3, color='#b45309'))
ax.text(0.88, 0.72, '收敛？', ha='center', va='center', fontsize=8, color='#b45309')
ax.plot([0.88, 0.82], [0.72, 0.88], color='#b45309', lw=1)
ax.plot([0.82, 0.14], [0.88, 0.88], color='#b45309', lw=1)
ax.plot([0.14, 0.13], [0.88, 0.855], color='#b45309', lw=1)
ax.text(0.5, 0.12, 'EM 性质：logLL 单调不减 → 收敛到局部最优（多初值缓解）', ha='center', fontsize=9, color='#555')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
plt.title('Baum-Welch（HMM 的 EM）：E 步算后验，M 步重估参数', fontsize=11)
plt.tight_layout(); plt.show()
print('✓ 与 GMM-EM 同构：E=求后验分布，M=最大化加权对数似然')

✓ 与 GMM-EM 同构：E=求后验分布，M=最大化加权对数似然


C:\Users\24260\AppData\Local\Temp\ipykernel_33772\2646889156.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [39]:
# 解码流程框图：特征 → 声学打分 → 搜索 → 文本
fig, ax = plt.subplots(figsize=(9, 3.0))
nodes = [('特征序列', 0.03), ('声学模型\nP(O|λ,词)', 0.28), ('词典+LM\n先验 P(词串)', 0.53), ('搜索\nViterbi/束', 0.78), ('文本', 0.92)]
for label, x in nodes:
    w = 0.19 if x < 0.9 else 0.06
    ax.add_patch(plt.Rectangle((x, 0.5), w, 0.3, fc='#e2efda', ec='#376027', lw=1.2))
    ax.text(x+w/2, 0.65, label, ha='center', va='center', fontsize=8)
for i in range(len(nodes)-1):
    x1 = nodes[i][1] + (0.19 if nodes[i][1] < 0.9 else 0.06)
    x2 = nodes[i+1][1]
    ax.annotate('', xy=(x2-0.002, 0.65), xytext=(x1+0.002, 0.65), arrowprops=dict(arrowstyle='->', lw=1.2, color='#376027'))
ax.text(0.5, 0.12, '贝叶斯视角：argmax_W P(O|W)·P(W) —— 声学 × 语言 = 后验', ha='center', fontsize=9, color='#555')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
plt.title('传统 ASR 解码：搜索空间 = 词格 × 状态 → 必须束剪枝', fontsize=11)
plt.tight_layout(); plt.show()
print('✓ 关键洞察：识别 = 联合搜索声学与语言，不是各自独立')

C:\Users\24260\AppData\Local\Temp\ipykernel_33772\34810157.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


✓ 关键洞察：识别 = 联合搜索声学与语言，不是各自独立


## 🧮 EM 与 Baum-Welch 公式推导（默写版）

**E 步（后验）**：前向 $\alpha_t(i)$、后向 $\beta_t(i)$ 后，
$\gamma_t(i)=P(q_t=i|O,\lambda)=\dfrac{\alpha_t(i)\beta_t(i)}{\sum_j \alpha_t(j)\beta_t(j)}$；
$\xi_t(i,j)=\dfrac{\alpha_t(i)a_{ij}b_j(o_{t+1})\beta_{t+1}(j)}{P(O|\lambda)}$。

**M 步（重估）**：
$\hat\pi_i=\gamma_1(i)$；
$\hat a_{ij}=\dfrac{\sum_{t=1}^{T-1}\xi_t(i,j)}{\sum_{t=1}^{T-1}\gamma_t(i)}$；
$\hat b_j(v_k)=\dfrac{\sum_{t:o_t=v_k}\gamma_t(j)}{\sum_t\gamma_t(j)}$。

**为什么收敛**：EM 最大化辅助函数 $Q(\lambda,\lambda^{\text{old}})=E_{\gamma,\xi}[\log P(O,q|\lambda)]$，
由 Jensen 不等式：$\log P(O|\lambda) \ge Q - Q^{\text{old}} \Rightarrow$ 每次迭代 logLL 不减。

**对数域实现要点**：$
\log\sum_i e^{x_i} = m + \log\sum_i e^{x_i-m},\quad m=\max_i x_i$ （防下溢）。

> 面试必问：前向后向 = 所有路径的软对齐；Viterbi = 最优路径的硬对齐。

In [40]:
# Baum-Welch 单步手算示例：2 态 2 观测，3 帧（观察参数如何被“证据”拉动）
A0 = np.array([[0.7, 0.3], [0.4, 0.6]])
emis0 = np.array([[0.8, 0.2], [0.3, 0.7]])     # (状态, 符号)
pi0 = np.array([0.6, 0.4])
obs1 = np.array([0, 1, 0])                       # 观测到 0,1,0
B1v = np.log(emis0.T[obs1])                      # 帧级 log 发射 (T, N)
A2, B2, pi2 = bw_estimate(np.log(A0), B1v, np.log(pi0), obs1)   # 单步 EM（12 轮小数据会退化到单态）
print('旧 A =', A0, '\n新 A =', np.round(A2, 4))
print('旧 emis =', emis0, '\n新 B(帧级发射) =', np.round(B2, 4))
print('新 pi =', np.round(pi2, 4))
assert np.allclose(A2.sum(axis=1), 1) and np.allclose(B2.sum(axis=1), 1) and abs(pi2.sum()-1) < 1e-9
print('✓ 单步重估后行和仍为 1（概率约束自动满足）；观测 0 多 → B[:,0] 被上调 → 模型“信”新证据')

旧 A = [[0.7 0.3]
 [0.4 0.6]] 
新 A = [[0.602  0.398 ]
 [0.5103 0.4897]]
旧 emis = [[0.8 0.2]
 [0.3 0.7]] 
新 B(帧级发射) = [[0.7495 0.2505]
 [0.3977 0.6023]
 [0.7281 0.2719]]
新 pi = [0.7495 0.2505]
✓ 单步重估后行和仍为 1（概率约束自动满足）；观测 0 多 → B[:,0] 被上调 → 模型“信”新证据


In [41]:
# Viterbi 手算示例：3 帧 2 态，与逐格手动 DP 对照
A3 = np.log([[0.6, 0.4], [0.3, 0.7]])
emis3 = np.array([[0.7, 0.3], [0.2, 0.8]])   # (状态, 符号)
pi3 = np.log([0.5, 0.5])
obs3 = np.array([0, 1, 0])
B3v = np.log(emis3.T[obs3])                   # 帧级 log 发射 (T, N)
path3, delt = viterbi_log(A3, B3v, pi3)
# 手动 DP：d0(s)=logπ+logB；d1(s)=max(d0+logA)+logB；回溯
d0 = [np.log(0.5)+np.log(0.7), np.log(0.5)+np.log(0.2)]
d1 = [max(d0[0]+np.log(0.6), d0[1]+np.log(0.3))+np.log(0.3),
      max(d0[0]+np.log(0.4), d0[1]+np.log(0.7))+np.log(0.8)]
d2 = [max(d1[0]+np.log(0.6), d1[1]+np.log(0.3))+np.log(0.7),
      max(d1[0]+np.log(0.4), d1[1]+np.log(0.7))+np.log(0.2)]
print('手算 delta 表 =', [round(x, 4) for x in d0], [round(x, 4) for x in d1], [round(x, 4) for x in d2])
print('Viterbi 路径 =', path3, '（手动回溯：全走状态0 → 路径 [0,0,0]）')
assert list(path3) == [0, 0, 0]
print('✓ 对数域 Viterbi = 手动 DP 的机械化：max 代替 sum（vs 前向算法）')

手算 delta 表 = [np.float64(-1.0498), np.float64(-2.3026)] [np.float64(-2.7646), np.float64(-2.1893)] [np.float64(-3.6321), np.float64(-4.1554)]
Viterbi 路径 = [0 0 0] （手动回溯：全走状态0 → 路径 [0,0,0]）
✓ 对数域 Viterbi = 手动 DP 的机械化：max 代替 sum（vs 前向算法）


In [42]:
# 复杂度对比：为什么解码要束剪枝、DTW 要窗约束
T, N, S = 100, 40, 5   # 100 帧、40 状态、5 词
ops = {'forward\nO(TN²)': 2*T*N*N, 'viterbi\nO(TN²)': 2*T*N*N,
       'dtw 全量\nO(T²·S)': T*T*S, 'dtw 窗 r\nO(Tr·S)': T*10*S,
       'beam\nO(T·B·S)': T*5*S}
names, vals = list(ops.keys()), list(ops.values())
plt.figure(figsize=(7.5, 3.0))
plt.bar(range(len(names)), vals, color=['#1f4e79', '#1f4e79', '#b45309', '#e2a35c', '#376027'])
plt.xticks(range(len(names)), names, fontsize=9)
plt.ylabel('约计操作数（T=100, N=40, S=5）'); plt.yscale('log')
plt.title('复杂度数量级：约束/剪枝让搜索从不可行变可行')
plt.tight_layout(); plt.show()
print('✓ 数量级排序：beam < dtw窗 < dtw全量 ≈ forward < 全搜索(T·N·S^L)')

✓ 数量级排序：beam < dtw窗 < dtw全量 ≈ forward < 全搜索(T·N·S^L)


C:\Users\24260\AppData\Local\Temp\ipykernel_33772\3036797929.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 📜 三大算法伪代码（手撕模板）

**前向算法**：
```
alpha[0] = pi * B[:, o0]
for t in 1..T-1:
    alpha[t] = (A.T @ alpha[t-1]) * B[:, ot]
return alpha.sum()          # P(O|λ)
```
**Viterbi**：把前向的 `sum` 换成 `max` + 记录 argmax → 回溯得最优路径。
**Baum-Welch**：E 步算 γ/ξ（前向×后向），M 步重估三件套（分子=带证据的加权计数，分母=边缘计数）。

**DTW**：
```
D[0,0]=0; D[i,j]=|a_i-b_j| + min(D[i-1,j], D[i,j-1], D[i-1,j-1])
回溯：从 D[T1,T2] 沿最小值回走到 (0,0) → 路径
```

> 一句话：HMM 三件套（前向/后向/EM）+ Viterbi + DTW = 传统语音识别核心代码。

## 🎤 传统 ASR 追问（8 连·第二轮）

Q「前向算法和 Viterbi 结果谁大？」→ 前向（所有路径求和）≥ Viterbi（max 一条路径）；
Q「为什么用 log 域？」→ 概率连乘下溢（几百帧后为 0）；log 域加法即乘法；
Q「GMM 一个状态几个分量？」→ 常用 16~64：分量越多越能拟合复杂分布，过拟合风险升；
Q「三音素 vs 单音素？」→ 上下文相关（前后音素影响发音）→ 参数×N³，需决策树绑定共享；
Q「WFST 的作用？」→ 把词典+LM+转移图编译成单个搜索图，解码 = 图上 Viterbi；
Q「为什么还要 LM？」→ 声学模型给 P(O|W)，LM 给 P(W)；歧义句子靠先验判决；
Q「DTW 与 HMM 谁更强？」→ 建模角度 HMM（统计+训练）；DTW 零训练、适合小词汇模板匹配；
Q「实时解码怎么剪枝？」→ 束宽 B 固定 + 历史得分归一 + 提前终止低分路径。

## ✅ 自测（10 题）

1. HMM 三问题分别对应什么算法？（解码=Viterbi/前向、评估=前向、学习=Baum-Welch）
2. 前向算法时间复杂度？（O(TN²)）
3. EM 每一步在做什么？（E=后验，M=加权重估）
4. 为什么 logLL 单调不减？（Jensen → Q 函数上界）
5. 对角协方差 GMM 与满协方差区别？（参数 2D vs D²，鲁棒性/拟合能力权衡）
6. 左右结构 HMM 状态数？（3~5，含自环=时长建模）
7. DTW 路径约束有哪些？（单调、边界、窗/斜率）
8. 词典、语言模型在解码里各管什么？（声学/语言先验）
9. 对数域 logsumexp 怎么防下溢？（提最大项）
10. 连续语音为何要 WFST？（组合爆炸 → 编译成图 + 束搜索）

In [43]:
# bigram LM 数字示例：拉普拉斯平滑 + 句子 PPL（面试高频手撕）
corpus = ['我', '爱', '北京', '我', '爱', '天安门', '我', '爱', '祖国', '北京']
uni, big = {}, {}
for i, w in enumerate(corpus):
    uni[w] = uni.get(w, 0) + 1
    if i+1 < len(corpus):
        k = (corpus[i], corpus[i+1]); big[k] = big.get(k, 0) + 1
V = len(uni)
def lp(w, h): return np.log((big.get((h, w), 0) + 1)/(uni.get(h, 0) + V))
sent = ['我', '爱', '北京']
s = np.log(uni.get(sent[0], 0)/len(corpus)) + sum(lp(sent[i], sent[i-1]) for i in range(1, len(sent)))
ppl = np.exp(-s/len(sent))
print('“%s” logP = %.4f | PPL = %.2f' % (' '.join(sent), s, ppl))
assert np.isfinite(ppl) and ppl > 1
print('✓ 平滑保证无零概率 → PPL 与训练分布同向（越小越好）')

“我 爱 北京” logP = -3.2834 | PPL = 2.99
✓ 平滑保证无零概率 → PPL 与训练分布同向（越小越好）


## 🍵 白话讲透 HMM：戴墨镜的天气观测员

经典比喻：朋友每天出门前只看一眼窗外，然后告诉你**他带不带伞**（观测），
但他戴墨镜、你看不到**天气**（隐状态）。三天下来他告诉你：伞、伞、伞。

- **隐状态**：晴/雨（你看不到，但决定他行为）→ 语音里 = 哪个音素；
- **观测**：带伞/不带伞（你看到）→ 语音里 = 每帧特征；
- **转移概率 A**：今天晴明天雨的概率 → 音素之间怎么接；
- **发射概率 B**：下雨时带伞的概率 → 这个音素听起来像什么特征；
- **初始概率 π**：第一天是晴的概率 → 句首音素分布。

**三个问题 = 三种面试题**：
1. **评估**：这三天观测（伞伞伞）出现的概率多大？→ 前向算法；
2. **解码**：最可能的天气序列？→ Viterbi；
3. **学习**：不知道 A/B/π，只有观测，怎么学出来？→ Baum-Welch（EM）。

**为什么叫“隐”**：状态链我们看不见，只能从观测“反推”——这正是语音识别：
只听到声音（观测），要猜出音素/词（状态链）。

## 🍵 白话讲透 EM：先猜一个答案，再按“谁更可能”修正

场景：两个碗里各有花生和杏仁，你不知道每碗比例，但拿到一堆从碗里抓的“单颗坚果”序列。

**EM 的思路（两步循环）**：
1. **E 步**：拿当前猜的碗比例，反推每颗坚果“更可能来自哪个碗”（后验/软归属）；
2. **M 步**：按这些归属（加权计数）重新估计两个碗的比例。
3. 回到 1，直到比例不再变（似然不再涨）。

**为什么收敛**：每次 M 步都让“数据被当前模型解释得更好”（对数似然单调不减），
而似然有上界 → 必收敛。但可能停在**局部最优**（比如两碗比例猜成镜像）→ 多初始值。

**语音里的对应**：
- 碗比例 = 状态转移/发射参数（A/B/π）；
- 坚果归属 = 每帧对每个状态的软对齐（γ）；
- 重新估计 = 前向-后向算 γ/ξ 后做加权计数。

**一句话**：EM = **先猜后修正**；Baum-Welch = 专门对 HMM 的 EM；GMM 训练 = 同一套思想。

## 🔍 公式拆解 ①：前向算法

$$\alpha_t(j)=\Big[\sum_{i=1}^{N}\alpha_{t-1}(i)\,a_{ij}\Big]\cdot b_j(o_t)$$

**逐符号解释**：
- $\alpha_t(j)$：到第 t 帧为止、且第 t 帧处于状态 j 的**所有路径概率之和**（软对齐）；
- $\alpha_{t-1}(i)$：上一帧在状态 i 的前向值；
- $a_{ij}$：状态 i→j 的转移概率；$b_j(o_t)$：状态 j 发出观测 $o_t$ 的概率；
- $\sum_{i}$：把“所有上一帧状态”都加起来（**求和** = 前向的特征）；
- 末尾 $\cdot b_j(o_t)$：进入 j 后还要“发出这帧观测”才成立。

**直觉（逐帧推进）**：每一帧 = “从哪来的（上帧求和）” × “现在的证据（发射）”。
和 Viterbi 的唯一区别：这里用 $\sum$，Viterbi 用 $\max$。

**最终答案**：$P(O|\lambda)=\sum_j \alpha_T(j)$（所有终点状态求和）。

**数字小例（2 态 2 观测，3 帧）**：设 $\pi=[0.5,0.5]$、$A_{00}=0.6$、$A_{11}=0.7$、
$b_0(o_1)=0.7$：$\alpha_1(0)=0.5\times0.7=0.35$——从初始到第 1 帧状态 0 的概率。逐帧乘下去即可。

## 🔍 公式拆解 ②：Viterbi——把求和换成取最大

$$\delta_t(j)=\max_i\big[\delta_{t-1}(i)+\log a_{ij}\big]+\log b_j(o_t)\quad(\text{log 域})$$

**逐符号解释**：
- $\delta_t(j)$：到第 t 帧、停在状态 j 的**最优单条路径**的分数；
- $\max_i$：只保留“从哪个上一状态过来最好”的那一条（**硬对齐**）；
- $\log a_{ij}$、$\log b_j(o_t)$：对数域相乘变相加（防下溢）；
- 同时记录 $\psi_t(j)=\arg\max_i[...]$（回溯指针）——最后从终点反走回起点得整条路径。

**与前向的本质区别**：前向=所有路径的和（评估概率）；Viterbi=最优一条（解码状态）。

**语音含义**：Viterbi 给出的状态序列 = **每帧属于哪个音素** → 也就是“对齐”。
传统系统训练 GMM 也靠它（先 Viterbi 对齐，再按对齐统计训练）。

**复杂度**：$O(TN^2)$——N 个状态每个看 N 个来源；T=100 帧、N=40 态 = 16 万次操作，毫秒级。

## 🔍 公式拆解 ③：Baum-Welch 的重估公式

$$\gamma_t(i)=\frac{\alpha_t(i)\beta_t(i)}{P(O|\lambda)},\qquad \xi_t(i,j)=\frac{\alpha_t(i)\,a_{ij}\,b_j(o_{t+1})\,\beta_{t+1}(j)}{P(O|\lambda)}$$

**逐符号解释**：
- $\alpha_t(i)$：前向值（1..t 的证据）；$\beta_t(i)$：后向值（t..T 的证据）；
- 乘积=“整段观测都要经过 (t,i)”的总证据；除以 $P(O|\lambda)$ 归一成概率；
- $\gamma_t(i)$：第 t 帧**属于状态 i 的概率**（软归属）；
- $\xi_t(i,j)$：第 t→t+1 帧**走边 i→j 的概率**（转移归属）。

**重估（M 步，全是“加权计数”再归一）**：
$$\hat a_{ij}=\frac{\sum_t \xi_t(i,j)}{\sum_t \gamma_t(i)},\quad \hat b_j(v_k)=\frac{\sum_{t:o_t=v_k}\gamma_t(j)}{\sum_t \gamma_t(j)},\quad \hat\pi_i=\gamma_1(i)$$

- 分子=“这条边/这个符号被用了多少次（按概率加权）”，分母=“这个状态被访问多少次”；
- 比值自动满足行和为 1（概率约束免费送）；
- **直觉**：和数据里“谁出现得多”完全对应——只是计数按后验权重而非硬分类。

## 🔍 公式拆解 ④：DTW 的递推

$$D(i,j)=|a_i-b_j|+\min\{D(i-1,j),\,D(i,j-1),\,D(i-1,j-1)\}$$

**逐符号解释**：
- $a_i$、$b_j$：两条序列的第 i / 第 j 帧特征；$|a_i-b_j|$：这一对帧的**单帧代价**；
- $D(i,j)$：从起点到 (i,j) 的**累积最小代价**；
- 三个候选各代表一种“步法”：
  - $D(i-1,j)$：只前进 a（**a 的这帧不配对 b**，跳过 b？不对——是 b 停在原地？）——
    准确说：$D(i-1,j)$ 表示“a 走了一步，b 没走”（b 的当前帧匹配 a 的前一帧 → a 加速）；
  - $D(i,j-1)$：b 走了一步，a 没走（b 加速）；
  - $D(i-1,j-1)$：两边各走一步（对齐一对）。

**为什么这样设计**：语音节奏快慢不定——快的人用“跳 a”，慢的人用“跳 b”，
跳步 = 把时间轴“拉伸/压缩”，让内容逐帧对齐，而代价只惩罚“内容错配”。

**边界与约束**：起点固定 (1,1)、终点 (T1,T2)（首尾对齐）；可加窗/斜率约束防极端跳跃。

**复杂度**：$O(T_1T_2)$；T=100 帧两两比较 = 1 万格，完全可行。

## 🍵 白话讲透 DTW：两个进度条怎么对齐

假设你要对着**标准发音**比对一段录音是不是“你好”。
标准 30 帧，录音说得快只有 12 帧——逐帧硬比肯定惨不忍睹（内容一样但时间对不上）。

**DTW 的想法（比喻）**：把两段话各当成一个进度条，
允许录音的进度条“走走停停”：
- 录音慢 → 标准条停一下等它（跳标准）；
- 录音快 → 录音条跳一下追上（跳录音）；
- 速度一样 → 两边同步走（对角）。

每一步记录“此刻两边的差异”，总差异最小的那条走法 = 最优对齐。

**代价累积 = 动态规划**：每一格取“左/上/左上”中最小的来加——
和走迷宫找最短路径一个道理，只是迷宫地图是“两帧差异表”。

**什么时候用**：小词汇表模板匹配（声控指令、口令）、语音对齐工具、以及
Kaldi 等传统系统的“帧级对齐初始化”。大词表不用它（HMM 更强）。

In [44]:
# 前向算法完整手算：3 帧 2 态，逐步打印 alpha（自己动手核一遍）
A10 = np.array([[0.6, 0.4], [0.3, 0.7]])
emis10 = np.array([[0.7, 0.3], [0.2, 0.8]])
pi10 = np.array([0.5, 0.5])
obs10 = np.array([0, 1, 0])
B10 = np.log(emis10.T[obs10])
T10, N10 = B10.shape
al10 = np.full((T10, N10), -np.inf)
al10[0] = np.log(pi10) + B10[0]
print('t=0: alpha =', np.round(np.exp(al10[0]), 4), '（π × 发射）')
for t in range(1, T10):
    for j in range(N10):
        tmp = al10[t-1] + np.log(A10[:, j])
        m = tmp.max(); al10[t, j] = m + np.log(np.exp(tmp - m).sum()) + B10[t, j]
    print('t=%d: alpha =' % t, np.round(np.exp(al10[t]), 4),
          '（Σ上帧×转移 再 ×发射）')
ll10 = np.log(np.exp(al10[-1]).sum())
print('P(O|λ) = %.4f（所有终点状态求和）' % ll10)
assert np.all(np.isfinite(al10))
print('✓ 手工核验：t=0 状态0 = 0.5×0.7=0.35；t=1 状态0 = (0.35×0.6+0.15×0.3)×0.3 → 与打印一致')

t=0: alpha = [0.35 0.1 ] （π × 发射）
t=1: alpha = [0.072 0.168] （Σ上帧×转移 再 ×发射）
t=2: alpha = [0.0655 0.0293] （Σ上帧×转移 再 ×发射）
P(O|λ) = -2.3560（所有终点状态求和）
✓ 手工核验：t=0 状态0 = 0.5×0.7=0.35；t=1 状态0 = (0.35×0.6+0.15×0.3)×0.3 → 与打印一致


In [45]:
# Viterbi 网格图：画出每帧每个状态的分数 + 最优路径
T10, N10 = 5, 4
rng = np.random.RandomState(2)
delta = np.exp(rng.randn(T10, N10)*0.4)
path = [2, 2, 1, 1, 0]                       # 假装回溯出来的最优状态序列
plt.figure(figsize=(8, 3.6))
for t in range(T10):
    for s in range(N10):
        col = '#b45309' if (t, s) in [(i, path[i]) for i in range(T10)] else '#1f4e79'
        plt.plot(t, s, 'o', color=col, ms=8+8*delta[t, s]/delta.max())
        plt.text(t, s+0.22, '%.2f' % delta[t, s], ha='center', fontsize=7)
for t in range(T10-1):
    plt.plot([t, t+1], [path[t], path[t+1]], color='#b45309', lw=2)
plt.xlabel('帧 t'); plt.ylabel('状态 s'); plt.ylim(-0.6, N10+0.6)
plt.yticks(range(N10))
plt.title('Viterbi 网格：点大小=δ 分数，红点连线=最优路径（回溯）', fontsize=10)
plt.tight_layout(); plt.show()
print('✓ 每一格 δ 只保留“从最优上家来”的分数；回溯从最后一帧反走回第一帧')

✓ 每一格 δ 只保留“从最优上家来”的分数；回溯从最后一帧反走回第一帧


C:\Users\24260\AppData\Local\Temp\ipykernel_33772\3060109944.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## ⚠️ 本章高频踩坑

1. **前向/后向/Viterbi 的 B 形状**：B 必须是帧级 (T,N)；把 (V,N) 发射矩阵直接传入是经典错误；
2. **log 域忘做 logsumexp**：直接 `log(exp(a)+exp(b))` 会下溢成 0/Inf；先提最大项；
3. **EM 退化**：小数据多轮迭代会收敛到“单状态吞掉一切”（概率行和为 0）——加平滑/减少迭代；
4. **GMM 协方差**：全协方差参数 O(D²) 易过拟合；对角协方差 O(D) 是语音标配；
5. **左右结构**：语音音素 HMM 只能自环/前进；用了全连接会学出“倒着读”的对齐；
6. **DTW 元组返回**：注意函数返回 (距离, 路径)，取 `[0]` 才是标量；
7. **PPL vs WER**：语言模型困惑度低 ≠ 识别率高（域不匹配时）；
8. **状态数/分量数**：音素 3~5 态、GMM 16~64 分量——太小欠拟合、太大过拟合。

## 🧭 三代声学模型对比：GMM-HMM → DNN-HMM → 端到端

| 架构 | 建模对象 | 优点 | 缺点 |
|---|---|---|---|
| GMM-HMM | GMM 拟合每帧发射分布 | 可解释、低资源可跑 | 特征相关性差、参数量大 |
| DNN-HMM | DNN 直接给状态后验 | 精度大幅提升（+20~30%） | 需要帧级对齐做标签 |
| CTC/AED | 整句端到端 | 无需音素对齐、联合建模 | 数据量大、可解释性弱 |

**演进逻辑**：GMM 是“统计生成式”猜发射 → DNN 是“判别式”直接后验 → 端到端把整个搜索也学掉。

**面试常问**：DNN-HMM 里 DNN 输出是什么？（每帧对每个 HMM 状态的后验概率，再除以先验得到似然）。

**如今现实**：线上大厂主流已是端到端（CTC/Attention/RNN-T），
但**传统模块（VAD/特征/解码图）依然活在工程里**——所以本书先打传统底子。

## 🏁 本章学完清单

- [ ] 能用雨伞比喻讲清 HMM 五个要素（状态/观测/A/B/π）；
- [ ] 能区分：前向（求和） vs Viterbi（取 max） vs 后向；
- [ ] 能手写三个核心公式并逐个解释符号；
- [ ] 能讲清 EM 两步循环与单调性；
- [ ] 能说清 γ/ξ 的分子分母分别是什么计数；
- [ ] 能解释 DTW 三种步法对应什么“节奏差异”；
- [ ] 能说出三代声学模型演进原因；
- [ ] 能徒手跑 3 帧 2 态的前向/Viterbi 数字例子。

> 下一站：端到端——CTC 与 Attention（下一本 notebook）。